# Setup

In [ ]:
import pandas as pd
import requests
import os
import re
import pickle

from glob import glob
from collections import Counter

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data

## Downloading The Data

In [ ]:
parquet_urls = requests.get('https://huggingface.co/api/datasets/google/civil_comments/parquet/default').json()
parquet_urls

{'test': ['https://huggingface.co/api/datasets/google/civil_comments/parquet/default/test/0.parquet'],
 'train': ['https://huggingface.co/api/datasets/google/civil_comments/parquet/default/train/0.parquet',
  'https://huggingface.co/api/datasets/google/civil_comments/parquet/default/train/1.parquet'],
 'validation': ['https://huggingface.co/api/datasets/google/civil_comments/parquet/default/validation/0.parquet']}

In [ ]:
url_list = []

for split_cat in parquet_urls:
  url_list.extend(parquet_urls[split_cat])

len(url_list)

4

In [ ]:
os.makedirs('parquets')

In [ ]:
par_id = 0

for url in url_list:
  par_file = requests.get(url).content

  with open('parquets/{}.parquet'.format(par_id), 'wb') as f:
    f.write(par_file)

  par_id += 1

## Data Manipulation

In [ ]:
par_files = glob('parquets/*')

df = pd.concat([pd.read_parquet(par_file) for par_file in par_files])

In [ ]:
df.head()

,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,Jeff Sessions is another one of Trump's Orwell...,0.200000,0.0,0.000000,0.0,0.200000,0.0,0.0
1,I actually inspected the infrastructure on Gra...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
2,No it won't . That's just wishful thinking on ...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
3,Instead of wringing our hands and nibbling the...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
4,how many of you commenters have garbage piled ...,0.753846,0.0,0.046154,0.0,0.723077,0.0,0.0


In [ ]:
df.shape

(1999514, 8)

In [ ]:
df = pd.DataFrame(df['text'].values, columns=['text'])

In [ ]:
df = df.iloc[:10000, :]

In [ ]:
init_samples = df.shape[0]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
final_samples = df.shape[0]

if final_samples == init_samples:
  print('All sample space reserved')
else:
  print('{} samples were deleted'.format((init_samples - final_samples)))

6 samples were deleted


In [ ]:
def preprocess(text):
    text = text.strip()
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

In [ ]:
df['text'] = df['text'].apply(preprocess)

In [ ]:
tokens = []
for i in range(df.shape[0]):
  tokens.extend(df['text'].values[i])

word_counts = Counter(tokens)
print('there are {} different tokens'.format(len(word_counts)))

there are 29398 different tokens


In [ ]:
word_to_index = {word: i for i, word in enumerate(word_counts.keys())}
index_to_word = {i: word for word, i in word_to_index.items()}
vocab_size = len(word_to_index)

# Model

## Skipgram

In [ ]:
def init_skipgram(tokens, window_size=2):
    pairs = []
    for idx, center_word in enumerate(tokens):
        for w in range(-window_size, window_size + 1):
            context_idx = idx + w
            if w == 0 or context_idx < 0 or context_idx >= len(tokens):
                continue
            context_word = tokens[context_idx]
            pairs.append((center_word, context_word))
    return pairs

In [ ]:
pairs = init_skipgram(df['text'].values[0])

print(df['text'].values[0])
for pair in pairs:
  print(pair)

['jeff', 'sessions', 'is', 'another', 'one', 'of', 'trumps', 'orwellian', 'choices', 'he', 'believes', 'and', 'has', 'believed', 'his', 'entire', 'career', 'the', 'exact', 'opposite', 'of', 'what', 'the', 'position', 'requires']
('jeff', 'sessions')
('jeff', 'is')
('sessions', 'jeff')
('sessions', 'is')
('sessions', 'another')
('is', 'jeff')
('is', 'sessions')
('is', 'another')
('is', 'one')
('another', 'sessions')
('another', 'is')
('another', 'one')
('another', 'of')
('one', 'is')
('one', 'another')
('one', 'of')
('one', 'trumps')
('of', 'another')
('of', 'one')
('of', 'trumps')
('of', 'orwellian')
('trumps', 'one')
('trumps', 'of')
('trumps', 'orwellian')
('trumps', 'choices')
('orwellian', 'of')
('orwellian', 'trumps')
('orwellian', 'choices')
('orwellian', 'he')
('choices', 'trumps')
('choices', 'orwellian')
('choices', 'he')
('choices', 'believes')
('he', 'orwellian')
('he', 'choices')
('he', 'believes')
('he', 'and')
('believes', 'choices')
('believes', 'he')
('believes', 'and')

In [ ]:
def generate_word_pairs():
    for tokens in df['text']:
        yield from init_skipgram(tokens)

word_pairs = generate_word_pairs()

In [ ]:
def generate_index_pairs():
    for center, context in generate_word_pairs():
        yield (word_to_index[center], word_to_index[context])

index_pairs = generate_index_pairs()

## Embedding Model

In [ ]:
class Pyrrhotite(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Pyrrhotite, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, center_words):
        embeds = self.embeddings(center_words)
        out = self.output_layer(embeds)
        return out

In [ ]:
EMBEDDING_DIMENSION = 32
EPOCHS = 10
LEARNING_RATE = 0.01

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, 'running')

model = Pyrrhotite(vocab_size, EMBEDDING_DIMENSION).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    total_loss = 0

    for center_idx, context_idx in index_pairs:
        center_tensor = torch.tensor([center_idx], dtype=torch.long).to(device)
        context_tensor = torch.tensor([context_idx], dtype=torch.long).to(device)

        optimizer.zero_grad()
        output = model(center_tensor)
        loss = criterion(output, context_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch}, Loss: {total_loss:.4f}')

cuda running
Epoch 0, Loss: 20070171.2627
Epoch 1, Loss: 0.0000
Epoch 2, Loss: 0.0000
Epoch 3, Loss: 0.0000
Epoch 4, Loss: 0.0000
Epoch 5, Loss: 0.0000
Epoch 6, Loss: 0.0000
Epoch 7, Loss: 0.0000
Epoch 8, Loss: 0.0000
Epoch 9, Loss: 0.0000


In [ ]:
word_embeddings = model.embeddings.weight.data
print("Embedding for 'behold':", word_embeddings[word_to_index['behold']])

Embedding for 'behold': tensor([ 0.3787,  0.0903, -0.9778,  0.1862,  0.4520,  0.7556, -0.9942,  1.7504,
        -1.0644, -0.7533,  0.8377,  0.1999,  0.4275,  1.1653, -0.8488, -0.4420,
         0.3859, -0.2118, -1.2554, -0.3154,  1.8252,  0.6898, -0.3729, -0.2850,
        -0.2545, -0.6109,  1.7024, -0.8090, -0.2002,  0.0129,  2.0742, -0.6587],
       device='cuda:0')


# Save & Load

## Save

In [ ]:
base_path = '/content/drive/MyDrive/colabout/pyrrhotite'

In [ ]:
torch.save(model.state_dict(), '{}/pyrrhotite.pth'.format(base_path))

In [ ]:
with open('{}/vocab.pkl'.format(base_path), 'wb') as f:
    pickle.dump(word_to_index, f)

## Load

In [ ]:
with open('{}/vocab.pkl'.format(base_path), 'rb') as f:
    word_to_index = pickle.load(f)
index_to_word = {i: w for w, i in word_to_index.items()}
vocab_size = len(word_to_index)

In [ ]:
model = Pyrrhotite(vocab_size, 32)
model.load_state_dict(torch.load('{}/pyrrhotite.pth'.format(base_path)))
model.eval()

Pyrrhotite(
  (embeddings): Embedding(29398, 32)
  (output_layer): Linear(in_features=32, out_features=29398, bias=True)
)

In [ ]:
word_embeddings = model.embeddings.weight.data
print("Embedding for 'behold':", word_embeddings[word_to_index['behold']])

Embedding for 'behold': tensor([ 0.3787,  0.0903, -0.9778,  0.1862,  0.4520,  0.7556, -0.9942,  1.7504,
        -1.0644, -0.7533,  0.8377,  0.1999,  0.4275,  1.1653, -0.8488, -0.4420,
         0.3859, -0.2118, -1.2554, -0.3154,  1.8252,  0.6898, -0.3729, -0.2850,
        -0.2545, -0.6109,  1.7024, -0.8090, -0.2002,  0.0129,  2.0742, -0.6587])
